In [1]:
!nvidia-smi

Sat Mar  7 11:38:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/gdrive/MyDrive/path/to/resnet_croped_train_dataset.zip /content

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: cannot stat '/content/gdrive/MyDrive/path/to/resnet_croped_train_dataset.zip': No such file or directory


In [5]:
!unzip -q "/content/drive/MyDrive/resnet_croped_train_dataset.zip" -d /content/custom_data

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [18]:
dataset_path = "/content/custom_data/resnet_croped_train_dataset"

In [19]:
!pip install timm

In [21]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

In [22]:
dataset = datasets.ImageFolder(dataset_path, transform=transform)

print("Total Images:", len(dataset))
print("Total Classes:", len(dataset.classes))
print(dataset.classes[:10])

Total Images: 12091
Total Classes: 67
['Buffalo_Chhattisgarhi', 'Buffalo_Jaffarabadi', 'Buffalo_banni', 'Buffalo_bargur', 'Buffalo_bhadwari', 'Buffalo_chilika', 'Buffalo_gojri', 'Buffalo_kalahandi', 'Buffalo_luit', 'Buffalo_marathwada']


In [23]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [24]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [29]:
num_classes = len(dataset.classes)

model = timm.create_model(
    "resnetv2_50",
    pretrained=True,
    num_classes=num_classes
)

model = model.to(device)

In [44]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [ ]:
epochs = 40
best_acc = 0

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.3,
    patience=3,
    # verbose=True
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

for epoch in range(epochs):

    # ---------------- TRAIN ----------------
    model.train()
    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{epochs} Train Loss: {avg_loss:.4f}")


    # ---------------- VALIDATION ----------------
    model.eval()

    correct = 0
    total = 0
    val_loss = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    accuracy = 100 * correct / total

    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {accuracy:.2f}%\n")


    # ---------------- SAVE BEST MODEL ----------------
    if accuracy > best_acc:
        best_acc = accuracy
        torch.save(model.state_dict(), "best_model.pth")
        print("Best model saved!\n")


    # ---------------- LR SCHEDULER ----------------
    scheduler.step(accuracy)

Epoch 1/40 Train Loss: 1.1513
Validation Loss: 2.2376
Validation Accuracy: 57.13%

Best model saved!

Epoch 2/40 Train Loss: 1.0372
Validation Loss: 2.2440
Validation Accuracy: 57.34%

Best model saved!

Epoch 3/40 Train Loss: 1.0104
Validation Loss: 2.2613
Validation Accuracy: 56.26%

Epoch 4/40 Train Loss: 0.9928
Validation Loss: 2.2308
Validation Accuracy: 57.59%

Best model saved!

Epoch 5/40 Train Loss: 0.9764
Validation Loss: 2.2378
Validation Accuracy: 57.17%

Epoch 6/40 Train Loss: 0.9614
Validation Loss: 2.2791
Validation Accuracy: 57.79%

Best model saved!

Epoch 7/40 Train Loss: 0.9605
Validation Loss: 2.2759
Validation Accuracy: 56.84%

Epoch 8/40 Train Loss: 0.9471
Validation Loss: 2.3050
Validation Accuracy: 56.97%

Epoch 9/40 Train Loss: 0.9419
Validation Loss: 2.2799
Validation Accuracy: 56.88%

Epoch 10/40 Train Loss: 0.9330
Validation Loss: 2.2934
Validation Accuracy: 56.97%

Epoch 11/40 Train Loss: 0.9108
Validation Loss: 2.2529
Validation Accuracy: 58.37%

Best mode

In [33]:
torch.save(
    model.state_dict(),
    "/content/resnetv2_breed_classifier.pth"
)

print("Model saved in Colab")

Model saved in Colab


In [34]:
from google.colab import files
files.download("/content/resnetv2_breed_classifier.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
dataset = datasets.ImageFolder("/content/custom_data/resnet_croped_train_dataset")
classes = dataset.classes
print(classes)

['Buffalo_Chhattisgarhi', 'Buffalo_Jaffarabadi', 'Buffalo_banni', 'Buffalo_bargur', 'Buffalo_bhadwari', 'Buffalo_chilika', 'Buffalo_gojri', 'Buffalo_kalahandi', 'Buffalo_luit', 'Buffalo_marathwada', 'Buffalo_mehsana', 'Buffalo_murrah', 'Buffalo_nagpuri', 'Buffalo_nili-ravi', 'Buffalo_pandharpuri', 'Buffalo_surti', 'Buffalo_toda', 'Cow_Amritmahal', 'Cow_Ayrshire', 'Cow_Bargur', 'Cow_Dangi', 'Cow_Deoni', 'Cow_Gir', 'Cow_Hallikar', 'Cow_Hariana', 'Cow_Himachali Pahari', 'Cow_Kangayam', 'Cow_Kankrej', 'Cow_Kenkatha', 'Cow_Khariar', 'Cow_Khillari', 'Cow_Konkan Kapila', 'Cow_Kosali', 'Cow_Krishna_Valley', 'Cow_Ladakhi', 'Cow_Lakhimi', 'Cow_Malnad_gidda', 'Cow_Mewati', 'Cow_Nari', 'Cow_Nimari', 'Cow_Ongole', 'Cow_Poda Thirupu', 'Cow_Pulikulam', 'Cow_Punganur', 'Cow_Purnea', 'Cow_Rathi', 'Cow_Red kandhari', 'Cow_Red_Sindhi', 'Cow_Sahiwal', 'Cow_Shweta Kapila', 'Cow_Tharparkar', 'Cow_Umblachery', 'Cow_Vechur', 'Cow_bachaur', 'Cow_badri', 'Cow_bhelai', 'Cow_dagri', 'Cow_gangatari', 'Cow_gaolao',

In [41]:
import torch
import timm
from torchvision import transforms, datasets
from PIL import Image
import torch.nn.functional as F

# ---------- LOAD CLASSES FROM DATASET ----------
dataset = datasets.ImageFolder("/content/custom_data/resnet_croped_train_dataset")
classes = dataset.classes
num_classes = len(classes)

print("Total Classes:", num_classes)

# ---------- DEVICE ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- LOAD MODEL ----------
model = timm.create_model(
    "resnetv2_50",
    pretrained=False,
    num_classes=num_classes
)

model.load_state_dict(torch.load("resnetv2_breed_classifier.pth", map_location=device))
model = model.to(device)
model.eval()

# ---------- IMAGE TRANSFORM ----------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# ---------- TEST IMAGE ----------
img_path = "test2.jpg"   # change to your image path

image = Image.open(img_path).convert("RGB")
img = transform(image).unsqueeze(0).to(device)

# ---------- PREDICTION ----------
with torch.no_grad():
    outputs = model(img)
    probs = F.softmax(outputs, dim=1)

confidence, predicted = torch.max(probs, 1)

breed = classes[predicted.item()]

print("Predicted Breed:", breed)
print("Confidence:", round(confidence.item()*100,2), "%")

Total Classes: 67
Predicted Breed: Cow_Konkan Kapila
Confidence: 99.94 %
